# Lesson 14 · Train your own Phyll

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gkiri/tiny_repo/blob/main/tiny-llm/notebooks/L14-train-your-own.ipynb)

Train **Phyll** yourself: the course's chat model, 8,726,016 numbers, from random numbers, on the 217,798 chats it learned
from, with Phyll's own recipe.

**To run it:** choose **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.

| | Quick run (the default) | Full run (`QUICK = False` in section 7) |
|---|---|---|
| Steps | 2,000 | 18,000, as many as Phyll had |
| Time on the course's server (CPU) | about 35 minutes | Phyll's own training: about 3.3 hours |
| Held-back loss | about 2.5 | Phyll: 1.858 |

A GPU is faster: the notebook prints its own estimate. Your numbers will differ a little.
The data and this notebook come from the course's public repository, [gkiri/tiny_repo](https://github.com/gkiri/tiny_repo).

## 1 · Set up

Install the libraries (Colab already has them), then choose where to compute: a GPU if there is one, otherwise the CPU.

In [ ]:
%pip install -q -r https://raw.githubusercontent.com/gkiri/tiny_repo/main/tiny-llm/requirements.txt
# in Colab, ignore any note about restarting: these libraries are already there

In [ ]:
import copy, gzip, json, math, os, shutil, sys, time, urllib.request

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from tokenizers import Tokenizer

REPO = "https://raw.githubusercontent.com/gkiri/tiny_repo/main/tiny-llm/"   # the course's files


def download(name):
    """Read a file from the course's repository on GitHub, as bytes."""
    with urllib.request.urlopen(REPO + name) as response:
        return response.read()


device = "cuda" if torch.cuda.is_available() else "cpu"   # a GPU if there is one, otherwise the CPU
print("PyTorch", torch.__version__, "· computing on", device)
if device == "cpu":
    print("No GPU: training is slow. Choose Runtime → Change runtime type → T4 GPU, then Runtime → Run all.")

## 2 · Get the chats

**Phyll's chats:** each one is a short talk between a person and Phyll, the cactus, in Phyll's chat template:
`<|im_start|>` opens a turn and `<|im_end|>` closes it. The **held-back** chats are never trained on, so they show how Phyll
does on chats it has not seen.

In [ ]:
def read_chats(name):
    """Download a gzipped file of chats, one per line, and keep each chat's text."""
    lines = gzip.decompress(download(name)).decode("utf-8").splitlines()
    texts = []
    for line in lines:
        texts.append(json.loads(line)["text"])   # each line is {"text": …, "category": …}
    return texts


train_texts = read_chats("data/train.jsonl.gz")
held_back_texts = read_chats("data/eval.jsonl.gz")
print(f"{len(train_texts):,} training chats · {len(held_back_texts):,} held-back chats\n")
print(train_texts[0], "\n")
print(train_texts[2])    # a chat that starts with the windowsill line: days since water, and the weather

## 3 · Text becomes tokens

Phyll's **tokenizer** cuts text into pieces (tokens) from a vocabulary of 4,096 and gives each one an id: whole words, parts
of words, and the markers.

In [ ]:
tokenizer = Tokenizer.from_str(download("data/tokenizer.json").decode("utf-8"))   # Phyll's own tokenizer
vocab_size = tokenizer.get_vocab_size()     # 4,096 tokens
context_length = 128                        # the most tokens Phyll reads at once

example = tokenizer.encode(train_texts[0])
print(vocab_size, "tokens in the vocabulary\n")
print(example.tokens[:12])     # the pieces: Ġ marks a space before a word, Ċ a new line
print(example.ids[:12])        # their ids: 1 opens a turn, 2 closes it

In [ ]:
def tokenize(texts):
    """Every chat as a tensor of token ids (at most 128 of them)."""
    chats = []
    for text in texts:
        ids = tokenizer.encode(text).ids
        chats.append(torch.tensor(ids[:context_length]))
    return chats


train_chats = tokenize(train_texts)          # about half a minute
held_back_chats = tokenize(held_back_texts)
lengths = [len(chat) for chat in train_chats]
print(f"{sum(lengths):,} training tokens · the longest chat has {max(lengths)} tokens")

## 4 · Training examples

One example is one chat: `x` is every token but the last, and `y` every token but the first, so at each place `y` holds the
right next token. A batch is 32 chats. Shorter chats are filled up with **padding**, id 0, which the loss skips.

In [ ]:
batch_size = 32        # chats per training step


def make_batch(chats):
    """x: each chat without its last token · y: the same chat without its first token. Both padded with 0."""
    longest = 0
    for chat in chats:
        longest = max(longest, len(chat) - 1)
    x = torch.zeros(len(chats), longest, dtype=torch.long)    # all 0 (padding) to start with
    y = torch.zeros(len(chats), longest, dtype=torch.long)
    for row in range(len(chats)):
        n = len(chats[row]) - 1
        x[row, :n] = chats[row][:-1]
        y[row, :n] = chats[row][1:]
    return x.to(device), y.to(device)


x, y = make_batch(train_chats[:batch_size])
print("x:", tuple(x.shape), "· y:", tuple(y.shape), "  (chats, places)\n")
print("x[0]:", x[0][:10].tolist(), "…")
print("y[0]:", y[0][:10].tolist(), "…")

## 5 · The model

```
ids ─► token table + place table ─► 6 blocks: attention, then feed-forward ─► final norm ─► 4,096 scores
       what, and where                 look back     think                                 one per next token
```

This is `model.py`, written out for reading: `query`, `key` and `value` are its `qkv` in three tables, `mix` is `out`, and
`grow` and `shrink` are `up` and `down`.

In [ ]:
embed_size = 384                        # numbers in each token's row
num_heads = 6                           # attention heads
head_size = embed_size // num_heads     # 64 numbers per head
hidden_size = 768                       # the feed-forward's wider middle
num_blocks = 6                          # blocks, one after another
dropout_rate = 0.1                      # while training, 10% of the numbers are switched off at random

In [ ]:
class Attention(nn.Module):              # nn.Module: a PyTorch part with learned numbers inside
    """Each token looks back at the earlier ones and collects what it needs."""

    def __init__(self):                  # __init__ builds the part ("self" is the part itself)
        super().__init__()               # let PyTorch set the part up first
        # nn.Linear: a learned table of weights (384 × 384) plus a bias; it turns each row into a new row
        self.query = nn.Linear(embed_size, embed_size)   # what each token is looking for
        self.key = nn.Linear(embed_size, embed_size)     # what each token offers
        self.value = nn.Linear(embed_size, embed_size)   # what each token passes on
        self.mix = nn.Linear(embed_size, embed_size)     # combines the heads' findings
        self.dropout = nn.Dropout(dropout_rate)          # nn.Dropout: switches off random numbers, only in training

    def forward(self, x):                # forward runs when the part is called: attention(x)
        n = x.shape[1]                   # x: (chats, places, 384), so n = places
        q = self.query(x)                # (chats, places, 384)
        k = self.key(x)
        v = self.value(x)
        ones = torch.ones(n, n, device=x.device)
        later = torch.triu(ones, diagonal=1) == 1         # True above the diagonal: the later tokens

        heads = []
        for h in range(num_heads):                       # each head uses its own 64 of the 384 numbers
            start = h * head_size
            end = start + head_size
            qh = q[:, :, start:end]                      # (chats, places, 64)
            kh = k[:, :, start:end]
            vh = v[:, :, start:end]
            kh_turned = kh.transpose(1, 2)               # swap the last two axes: (chats, 64, places)
            scores = qh @ kh_turned                      # @ multiplies matrices: every query · every key
            scores = scores / math.sqrt(head_size)       # keep the scores small
            scores = scores.masked_fill(later, float("-inf"))   # -inf for later tokens: probability 0
            weights = F.softmax(scores, dim=-1)          # along the last axis: shares that add up to 1
            weights = self.dropout(weights)
            heads.append(weights @ vh)                   # a weighted mix of the values: (chats, places, 64)
        all_heads = torch.cat(heads, dim=-1)             # the six heads side by side: (chats, places, 384)
        return self.mix(all_heads)

In [ ]:
class FeedForward(nn.Module):
    """Each token's row on its own: widen, bend, narrow."""

    def __init__(self):
        super().__init__()
        self.grow = nn.Linear(embed_size, hidden_size)      # 384 → 768
        self.shrink = nn.Linear(hidden_size, embed_size)    # 768 → 384
        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, x):
        x = self.grow(x)
        x = F.relu(x)               # ReLU bends the numbers: negative ones become 0, the rest pass
        x = self.shrink(x)
        return self.dropout(x)


class Block(nn.Module):
    """Look back (attention), then think (feed-forward); each result is added to the row."""

    def __init__(self):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_size)     # LayerNorm: rescales each row to a steady size
        self.attention = Attention()
        self.norm2 = nn.LayerNorm(embed_size)
        self.feed_forward = FeedForward()

    def forward(self, x):
        x = x + self.attention(self.norm1(x))     # look back, and add what was found
        x = x + self.feed_forward(self.norm2(x))  # think, and add that too
        return x

In [ ]:
class Phyll(nn.Module):
    def __init__(self):
        super().__init__()
        # nn.Embedding: a table with one learned row per id
        self.token_table = nn.Embedding(vocab_size, embed_size)       # a row for each token
        self.place_table = nn.Embedding(context_length, embed_size)   # a row for each place 0 … 127
        self.dropout = nn.Dropout(dropout_rate)
        self.blocks = nn.ModuleList()                                 # nn.ModuleList: a list of parts PyTorch can see
        for _ in range(num_blocks):
            self.blocks.append(Block())
        self.final_norm = nn.LayerNorm(embed_size)
        self.readout = nn.Linear(embed_size, vocab_size, bias=False)  # a row → a score for each token
        self.readout.weight = self.token_table.weight                 # the readout reuses the token table's numbers

        # start with small random numbers (biases at 0), so the first guesses are nearly even
        for part in self.modules():                                   # visit every part, inside parts too
            if isinstance(part, (nn.Linear, nn.Embedding)):
                nn.init.normal_(part.weight, std=0.02)                # random numbers around 0
            if isinstance(part, nn.Linear) and part.bias is not None:   # (the readout has no bias)
                nn.init.zeros_(part.bias)

    def forward(self, ids):                                           # ids: (chats, places)
        places = torch.arange(ids.shape[1], device=ids.device)        # 0, 1, 2, …
        x = self.token_table(ids) + self.place_table(places)          # what + where: (chats, places, 384)
        x = self.dropout(x)
        for block in self.blocks:                                     # six blocks, one after another
            x = block(x)
        x = self.final_norm(x)
        scores = self.readout(x)                                      # (chats, places, 4,096)
        return scores

## 6 · Before training

The model starts with small random numbers, so it knows nothing yet. The **loss** measures its surprise at the right next
token: guessing evenly among 4,096 tokens gives ln 4,096 ≈ 8.32. `reply` lets it answer, one token at a time.

In [ ]:
torch.manual_seed(42)                # a seed fixes the random numbers: 42 is Phyll's
model = Phyll().to(device)


def count(part):
    """How many learned numbers a part has."""
    total = 0
    for numbers in part.parameters():
        total += numbers.numel()
    return total


print("token table   ", f"{count(model.token_table):>9,}   (the readout reuses it)")
print("place table   ", f"{count(model.place_table):>9,}")
print("6 blocks      ", f"{count(model.blocks):>9,}")
print("final norm    ", f"{count(model.final_norm):>9,}")
print(f"total          {count(model):>9,} learned numbers")

In [ ]:
def loss_of(scores, y):
    """Cross-entropy: how surprised the model is by the right next tokens (lower is better)."""
    flat_scores = scores.reshape(-1, vocab_size)     # one row of 4,096 scores per place (-1: work out how many)
    flat_targets = y.reshape(-1)                     # the right next token at each place
    return F.cross_entropy(flat_scores, flat_targets, ignore_index=0)   # ignore_index=0: skip the padding


def reply(model, message, temperature=0.7, seed=0):
    """The model's answer, one token at a time, picked the way the course's chat picks: keep the 50 best tokens,
    divide their scores by the temperature (below 1 is safer), and draw one. The same seed gives the same answer."""
    torch.manual_seed(seed)
    ids = tokenizer.encode("<|im_start|>user\n" + message + "<|im_end|>\n<|im_start|>assistant\n").ids
    answer = []
    model.eval()                                         # eval: dropout off while answering
    for _ in range(48):                                  # at most 48 tokens
        window = torch.tensor([ids[-context_length:]], device=device)
        with torch.no_grad():                            # only predicting: no gradients needed
            scores = model(window)[0, -1]                # the first chat, its last place: 4,096 scores
        top_scores, top_ids = torch.topk(scores / temperature, 50)   # the 50 best tokens
        probabilities = F.softmax(top_scores, dim=-1)
        pick = torch.multinomial(probabilities, 1).item()            # draw one of the 50
        next_id = top_ids[pick].item()                   # .item(): a tensor's number as plain Python
        if next_id == 2:                                 # 2 closes the turn: the answer is done
            break
        ids.append(next_id)
        answer.append(next_id)
    model.train()                                        # train: dropout back on
    return tokenizer.decode(answer)


x, y = make_batch(held_back_chats[:batch_size])
model.eval()
with torch.no_grad():
    loss = loss_of(model(x), y)
print(f"untrained loss {loss.item():.2f} · an even guess: ln {vocab_size:,} = {math.log(vocab_size):.2f}\n")
print("you: are you hungry\nmodel:", reply(model, "are you hungry"))

## 7 · The recipe

Phyll's recipe: the AdamW optimizer, and a learning rate that rises for 500 steps, then glides down along a cosine curve.
Every 500 steps, a **check** measures the loss and accuracy on the first 1,600 held-back chats, the way Phyll's training
did, so two checks differ only because the model changed.

In [ ]:
QUICK = True        # True: the first 2,000 steps · False: all 18,000 steps (use a GPU)
if QUICK:
    steps = 2_000
else:
    steps = 18_000

peak_lr = 0.0003        # the highest learning rate
final_lr = 0.00003      # the learning rate at the very end
warmup_steps = 500      # steps to rise from 0 to the peak
total_steps = 18_000    # the whole schedule (the quick run is its start)
check_every = 500       # steps between checks
check_batches = 50      # a check reads 50 batches of 32: the first 1,600 held-back chats


def learning_rate(step):
    if step < warmup_steps:                                       # rise in a straight line
        return peak_lr * step / warmup_steps
    progress = (step - warmup_steps) / (total_steps - warmup_steps)   # 0 → 1
    return final_lr + 0.5 * (peak_lr - final_lr) * (1 + math.cos(math.pi * progress))   # glide down

In [ ]:
def check(model, chats, batches):
    """The loss and accuracy on the first batches of chats, the way Phyll's training measured it:
    dropout off, and the loss is the average of the batches' losses."""
    model.eval()                                     # eval: dropout off while measuring
    total_loss = 0
    right = 0
    counted = 0
    with torch.no_grad():                            # only measuring, so no gradients
        for b in range(batches):
            x, y = make_batch(chats[b * batch_size : (b + 1) * batch_size])
            scores = model(x)
            total_loss += loss_of(scores, y).item()
            guesses = scores.argmax(dim=-1)          # the top-scoring token at each place
            real = y != 0                            # True where there is a token, False on padding
            correct = (guesses == y) & real          # & : both true (a right guess, on a real token)
            right += correct.sum().item()
            counted += real.sum().item()
    model.train()                                    # train: dropout back on
    return total_loss / batches, right / counted


print(f"{steps:,} steps · learning rate at step 0, 500, 2,000, 18,000:",
      [round(learning_rate(s), 6) for s in [0, 500, 2_000, 18_000]])

## 8 · Train

Each step: take a batch, predict, measure the loss, compute the **gradients** (how each number changes the loss), and let
AdamW move every number a little against its gradient. After each check, ★ marks a new best held-back loss, which is kept.

In [ ]:
torch.manual_seed(42)                 # Phyll's seed: the same starting numbers, the same batches
model = Phyll().to(device)
model.train()                         # train: dropout on
optimizer = torch.optim.AdamW(model.parameters(), lr=peak_lr, betas=(0.9, 0.95), weight_decay=0.1)
# betas: how much AdamW remembers of earlier steps · weight_decay: shrinks every number a little at each step

batches_per_pass = math.ceil(len(train_chats) / batch_size)   # a pass reads every training chat once
history = {"step": [], "train_loss": [], "held_back_loss": [], "held_back_accuracy": []}
losses = []                           # every training batch's loss
best_loss = float("inf")              # infinity: any first check is better

In [ ]:
start_time = time.time()
for step in range(steps + 1):
    if step % batches_per_pass == 0:
        order = torch.randperm(len(train_chats))                 # a new random order of the chats for each pass
    first = (step % batches_per_pass) * batch_size
    picked = order[first : first + batch_size]
    x, y = make_batch([train_chats[i] for i in picked])          # 1. a batch

    optimizer.param_groups[0]["lr"] = learning_rate(step)        # this step's learning rate
    scores = model(x)                                            # 2. predict
    loss = loss_of(scores, y)                                    # 3. the loss
    optimizer.zero_grad()                                        # clear the last step's gradients
    loss.backward()                                              # 4. the gradients
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)      # all gradients together too big? shrink them
    optimizer.step()                                             # 5. update every number
    losses.append(loss.item())

    if step > 0 and step % check_every == 0:                     # a check
        train_loss = sum(losses[-check_every:]) / check_every    # the last 500 training batches
        held_back_loss, accuracy = check(model, held_back_chats, check_batches)
        history["step"].append(step)
        history["train_loss"].append(train_loss)
        history["held_back_loss"].append(held_back_loss)
        history["held_back_accuracy"].append(accuracy)
        mark = ""
        if held_back_loss < best_loss:                           # a new best: keep a copy of its numbers
            best_loss, best_step, mark = held_back_loss, step, "★"
            best_numbers = copy.deepcopy(model.state_dict())
        print(f"step {step:6,} · training loss {train_loss:.4f} · held-back loss {held_back_loss:.4f}"
              f" · accuracy {accuracy:.1%} {mark}")

    if step == 100:
        minutes = (time.time() - start_time) / 100 * total_steps / 60
        print(f"(at this speed, all 18,000 steps take about {minutes:.0f} minutes)")

model.load_state_dict(best_numbers)                              # keep the best model
print(f"\nDone in {time.time() - start_time:.0f} seconds. Best held-back loss {best_loss:.4f} at step {best_step:,}.")

## 9 · Watch it learn

Your run, drawn over Phyll's own first run (the thick grey line): the same recipe, on an earlier set of chats, checked on
that set's held-back chats. The dashed line is the finished Phyll.

In [ ]:
phyll = json.loads(download("model/phyll-record.json"))   # Phyll's recorded training, and the finished Phyll's numbers
first_run = phyll["first_run"]
finished = phyll["finished"]

curves, (left, right) = plt.subplots(1, 2, figsize=(12, 4))
left.plot(first_run["step"], first_run["held_back_loss"], color="lightgrey", linewidth=6, label="Phyll's first run, held back")
left.axhline(finished["check_loss"], color="grey", linestyle="--", label="Phyll, finished")
left.plot(history["step"], history["train_loss"], "o-", label="your run, training chats")
left.plot(history["step"], history["held_back_loss"], "o-", label="your run, held-back chats")
left.set(title="Loss: lower is better", xlabel="step", xlim=(0, steps))
left.legend()
right.plot(history["step"], [a * 100 for a in history["held_back_accuracy"]], "o-", color="tab:green", label="your run")
right.axhline(finished["check_accuracy"] * 100, color="grey", linestyle="--", label="Phyll, finished")
right.set(title="Accuracy on held-back chats: higher is better", xlabel="step", ylabel="% right", xlim=(0, steps))
right.legend()
plt.show()

print("step     your held-back loss   Phyll's first run")
for i in range(len(history["step"])):
    step = history["step"][i]
    if step in first_run["step"]:
        j = first_run["step"].index(step)
        print(f"{step:6,}   {history['held_back_loss'][i]:19.3f}   {first_run['held_back_loss'][j]:17.3f}")

## 10 · How good is it?

The kept model reads every held-back chat, measured like the checks, and is compared with an even guess and the finished
Phyll, measured the same way.

In [ ]:
all_batches = math.ceil(len(held_back_chats) / batch_size)        # every held-back chat
final_loss, final_accuracy = check(model, held_back_chats, all_batches)
print(f"All {len(held_back_chats):,} held-back chats: loss {final_loss:.3f} · accuracy {final_accuracy:.1%}\n")

even = math.log(vocab_size)
print(f"an even guess   loss {even:.3f} · accuracy {1 / vocab_size:.2%}")
print(f"your model      loss {final_loss:.3f} · accuracy {final_accuracy:.1%}")
print(f"Phyll           loss {finished['all_loss']:.3f} · accuracy {finished['all_accuracy']:.1%}")

names = ["an even guess", "your model", "Phyll"]
losses_to_compare = [even, final_loss, finished["all_loss"]]
bars = plt.figure(figsize=(7, 3.5))
plt.bar(names, losses_to_compare, color=["lightgrey", "tab:orange", "tab:green"])
for i in range(len(names)):
    plt.text(i, losses_to_compare[i] + 0.1, f"{losses_to_compare[i]:.2f}", ha="center")
plt.ylim(0, max(losses_to_compare) + 1)
plt.title("Loss on every held-back chat: lower is better")
plt.show()

## 11 · Talk to it

The course's question first: **are you hungry?** The finished Phyll starts its answer with *yes* almost every time. Then
three questions, with the finished Phyll's answers beside yours (the same seed), and one of your own.

In [ ]:
prompt = tokenizer.encode("<|im_start|>user\nare you hungry<|im_end|>\n<|im_start|>assistant\n").ids
model.eval()
with torch.no_grad():
    scores = model(torch.tensor([prompt], device=device))[0, -1]   # the scores for the answer's first token
model.train()
probabilities = F.softmax(scores, dim=-1)
yes = probabilities[tokenizer.token_to_id("yes")].item()
print(f"'yes' as the first token: your model {yes:.1%} · Phyll {finished['yes']:.1%}\n")

replies = ""
for question in phyll["questions"]:
    answer = reply(model, question, seed=1)
    replies += f"you:        {question}\nyour model: {answer}\nPhyll:      {phyll['replies'][question]}\n\n"
print(replies)

In [ ]:
my_message = "do you like the rain"     # try your own (Phyll's chats are in lower case)
answer = reply(model, my_message, seed=1)
replies += f"you:        {my_message}\nyour model: {answer}\n\n"   # saved in section 12
print(answer)

## 12 · Save your model

Everything goes into the `outputs/` folder: in Colab, click the folder icon on the left. Colab also downloads it as a zip.

In [ ]:
os.makedirs("outputs", exist_ok=True)
torch.save(model.state_dict(), "outputs/model.pt")                  # the trained numbers
tokenizer.save("outputs/tokenizer.json")                            # the tokenizer, to turn text into ids again
info = {"vocab_size": vocab_size, "context_length": context_length, "embed_size": embed_size, "num_heads": num_heads,
        "hidden_size": hidden_size, "num_blocks": num_blocks, "dropout_rate": dropout_rate, "steps": steps,
        "best_step": best_step, "held_back_loss": final_loss, "held_back_accuracy": final_accuracy}
with open("outputs/model_info.json", "w") as f:
    json.dump(info, f, indent=1)
with open("outputs/history.json", "w") as f:
    json.dump(history, f, indent=1)
with open("outputs/replies.txt", "w") as f:
    f.write(replies)
curves.savefig("outputs/training_curves.png")
bars.savefig("outputs/comparison.png")

reloaded = Phyll().to(device)                                       # rebuild it from the file
reloaded.load_state_dict(torch.load("outputs/model.pt", map_location=device, weights_only=True))
same_loss, _ = check(reloaded, held_back_chats, check_batches)
assert abs(same_loss - best_loss) < 1e-4
print("reloaded outputs/model.pt: same held-back loss ✓\n")
for name in sorted(os.listdir("outputs")):
    print(" ", name)

In [ ]:
shutil.make_archive("my-phyll", "zip", "outputs")
if "google.colab" in sys.modules:                 # in Colab, download the zip
    from google.colab import files
    files.download("my-phyll.zip")

## 13 · Try next

Change one thing, then **Runtime → Run all**:

- **The full run:** `QUICK = False` in section 7, on a GPU. Phyll's own 18,000 steps took about 3.3 hours on the course's
  server, without a GPU.
- **A smaller Phyll:** `num_blocks = 3` in section 5: fewer numbers, faster steps.
- **No dropout:** `dropout_rate = 0`, then compare the training and held-back losses.
- **The windowsill line:** ask `[days 3 | weather rain] should i water now`.
- **Your own chats:** a file in the same format, one `{"text": …}` per line, in place of section 2's.

After a change, your curve leaves Phyll's grey line. That is your experiment.